# AXE 5 — Graphe Social Instagram

**Objectif** : construire un graphe de relations à partir des messages privés Instagram.

**Sources**
- `data/raw/INSTAGRAM/your_instagram_activity/messages/inbox/` — conversations

**Output** : `warehouse/social_graph.parquet`

In [1]:
import json
import os
import re

import pandas as pd

In [2]:
# ─── CHEMINS ─────────────────────────────────────────────────────────────────
if os.path.exists("/app/warehouse"):
    BASE = "/app"
else:
    _candidate = os.path.abspath(".")
    for _ in range(6):
        if os.path.isdir(os.path.join(_candidate, "warehouse")):
            break
        _candidate = os.path.dirname(_candidate)
    BASE = _candidate

INBOX     = os.path.join(BASE, "data/raw/INSTAGRAM/your_instagram_activity/messages/inbox")
WAREHOUSE = os.path.join(BASE, "warehouse")
os.makedirs(WAREHOUSE, exist_ok=True)

MIN_MESSAGES = 5  # seuil pour exclure les inconnus / spams

print("BASE:", BASE)
print("Inbox conversations:", len(os.listdir(INBOX)) if os.path.exists(INBOX) else "N/A")

BASE: /opt/spark
Inbox conversations: 402


In [3]:
# ─── CLOSE FRIENDS MANUELS ───────────────────────────────────────────────────
# Les noms de dossiers inbox ne correspondent pas aux usernames Instagram.
# Renseigner ici les noms de dossiers exacts (partie avant le _ID numérique).
# Exemple : dossier 'nana_18068525429472883' → mettre 'nana'
#           dossier 'alice_1093935518743724' → mettre 'alice_1093935518743724' si ambiguïté

CLOSE_FRIENDS = {
    "_louange___",
    "_paulinalambin",
    "adamvdd",
    "clara",
    "djyoyo",
    "erwan",
    "evan",
    "fafie",
    "fredfnmz",
    "gabi",
    "hugobernard",
    "jen",
    "jerome",
    "laura",
    "lenysoetaerts",
    "lou",
    "maelle",
    "manonvandy",
    "pilou",
    "romane",
    "sachou",
    "vic",
    "zo",
    "loulou",
    "nana",
    "3li0tttt"
}

CLOSE_FRIENDS_MULTIPLIER = 2.0

print(f"Close friends configurés : {len(CLOSE_FRIENDS)}")

Close friends configurés : 26


In [4]:
# ─── PARSING DE L'INBOX ──────────────────────────────────────────────────────
def _parse_conversation(conv_dir: str, folder: str) -> dict | None:
    msg_files = sorted(
        [f for f in os.listdir(conv_dir) if f.startswith("message_") and f.endswith(".json")],
        key=lambda f: int(re.search(r'(\d+)', f).group(1)),
        reverse=True,
    )
    if not msg_files:
        return None

    with open(os.path.join(conv_dir, msg_files[0]), encoding="utf-8") as f:
        data = json.load(f)

    if len(data.get("participants", [])) != 2:
        return None

    msg_count = len(data.get("messages", []))
    if msg_count < MIN_MESSAGES:
        return None

    # label = partie avant l'ID numérique (peut être en doublon entre dossiers)
    # node_id = nom de dossier complet → toujours unique
    label = re.split(r'_\d{10,}', folder)[0].lower()
    node_id = folder.lower()
    is_close = (label in CLOSE_FRIENDS) or (node_id in CLOSE_FRIENDS)

    return {"node_id": node_id, "label": label, "message_count": msg_count, "in_close_friends": is_close}


records = []
for folder in os.listdir(INBOX):
    conv_dir = os.path.join(INBOX, folder)
    if not os.path.isdir(conv_dir):
        continue
    result = _parse_conversation(conv_dir, folder)
    if result:
        records.append(result)

df = pd.DataFrame(records)
df["weight"] = df.apply(
    lambda row: row["message_count"] * CLOSE_FRIENDS_MULTIPLIER if row["in_close_friends"] else float(row["message_count"]),
    axis=1,
)
df = df.sort_values("weight", ascending=False).reset_index(drop=True)

print(f"Conversations retenues (>= {MIN_MESSAGES} msgs) : {len(df)}")
print(f"Close friends : {df['in_close_friends'].sum()}")
print()
print(df[["label", "message_count", "in_close_friends", "weight"]].head(30).to_string())

Conversations retenues (>= 5 msgs) : 128
Close friends : 26

               label  message_count  in_close_friends   weight
0             loulou           9920              True  19840.0
1                lou           6685              True  13370.0
2                jen           6506              True  13012.0
3               gabi           6502              True  13004.0
4           3li0tttt           5306              True  10612.0
5              alice           9726             False   9726.0
6               nana           4809              True   9618.0
7              pilou           4502              True   9004.0
8             djyoyo           2506              True   5012.0
9              laure           4974             False   4974.0
10            sachou           2258              True   4516.0
11            maelle           2257              True   4514.0
12        manonvandy           2032              True   4064.0
13            mylene           3695             False   3

In [5]:
# ─── SAUVEGARDE ──────────────────────────────────────────────────────────────
out_path = os.path.join(WAREHOUSE, "social_graph.parquet")
df.to_parquet(out_path, index=False)
print(f"Sauvegardé : {out_path} ({len(df)} lignes, {df['message_count'].sum():,} msgs total)")

Sauvegardé : /opt/spark/warehouse/social_graph.parquet (128 lignes, 97,499 msgs total)
